In [1]:
from ingest import load_faq_data
documents = load_faq_data(file_path="../documents/all_documents.json") # adjust path as needed

In [2]:
documents[10]

{'course': 'data-engineering',
 'section': 'General Course-Related Questions',
 'question': 'Office Hours: I can’t attend the “Office hours” / workshop, will it be recorded?',
 'answer': 'Yes! Every "Office Hours" will be recorded and available a few minutes after the live session is over; so you can view (or rewatch) whenever you want.',
 'doc_id': 'a411de5004'}

In [3]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm":
        documents_llm.append(doc)

len(documents_llm)

111

In [4]:
documents = documents_llm

In [6]:
doc = documents[1]
print(doc["doc_id"])
print(doc["question"])
print(doc["answer"])

977bf7786c
Course: I have registered for the LLM . When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.


In [7]:
doc

{'course': 'llm',
 'section': 'General Course-Related Questions',
 'question': 'Course: I have registered for the LLM . When can I expect to receive the confirmation email?',
 'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.",
 'doc_id': '977bf7786c'}

In [8]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [9]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [ ]:
data_gen_instructions

In [ ]:
from openai import OpenAI

openai_client = OpenAI(
    base_url="https://dashscope-intl.aliyuncs.com/compatible-mode/v1",
    api_key="your-key"
)

In [14]:
import json
user_prompt = json.dumps(doc)
user_prompt

'{"course": "llm", "section": "General Course-Related Questions", "question": "Course: I have registered for the LLM . When can I expect to receive the confirmation email?", "answer": "You don\'t need it. You\'re accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.", "doc_id": "977bf7786c"}'

In [15]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [16]:
response = openai_client.responses.create(
    model="deepseek-v4-flash",
    # model="qwen-max",  # <--- Change this
    input=messages,
    # text_format=Questions
)

In [19]:
response.output_text

'1. I registered for the LLM course a couple of days ago, but I still haven’t seen any confirmation email. Should I contact support or just wait it out?  \n2. Do I have to complete the registration process before I can start watching the lectures and turning in homework, or can I just jump right in?  \n3. Is there a hard deadline for signing up? What happens if I never officially register—will I lose access to the course materials or be blocked from submitting work?  \n4. I filled out the registration form, but I’m not sure if it actually went through since no email came. Can I still do the assignments and get feedback without that confirmation?  \n5. How important is the registration step for this course? Is it purely to help the team plan things, or does it determine whether I’m considered an official participant?'

In [18]:
doc

{'course': 'llm',
 'section': 'General Course-Related Questions',
 'question': 'Course: I have registered for the LLM . When can I expect to receive the confirmation email?',
 'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.",
 'doc_id': '977bf7786c'}

In [20]:
from evaluation_utils import llm_structured

In [23]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions,
    model="deepseek-v4-flash"
)


In [25]:
result.questions

["I signed up for the LLM course but haven't gotten a confirmation email. Is that normal?",
 'Do I really need to register for the course to access the materials and submit homework?',
 "I registered but didn't get any email. Should I be worried that my registration didn't go through?",
 "If I don't register, can I still join the course and submit assignments?",
 "What's the point of registering if it's not required and I won't get a confirmation?"]

In [24]:
for i in (result.questions):
    print(i)

I signed up for the LLM course but haven't gotten a confirmation email. Is that normal?
Do I really need to register for the course to access the materials and submit homework?
I registered but didn't get any email. Should I be worried that my registration didn't go through?
If I don't register, can I still join the course and submit assignments?
What's the point of registering if it's not required and I won't get a confirmation?


In [26]:
usage

CompletionUsage(completion_tokens=518, prompt_tokens=261, total_tokens=779, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=400, rejected_prediction_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=None, cached_tokens=0))

In [27]:
from evaluation_utils import calc_price

In [28]:
calc_price(usage)

{'input_cost': 2.61e-05,
 'output_cost': 0.00020720000000000002,
 'total_cost': 0.00023330000000000003}

In [29]:
doc

{'course': 'llm',
 'section': 'General Course-Related Questions',
 'question': 'Course: I have registered for the LLM . When can I expect to receive the confirmation email?',
 'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.",
 'doc_id': '977bf7786c'}

In [30]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["doc_id"]
    })

records

[{'question': "I signed up for the LLM course but haven't gotten a confirmation email. Is that normal?",
  'document': '977bf7786c'},
 {'question': 'Do I really need to register for the course to access the materials and submit homework?',
  'document': '977bf7786c'},
 {'question': "I registered but didn't get any email. Should I be worried that my registration didn't go through?",
  'document': '977bf7786c'},
 {'question': "If I don't register, can I still join the course and submit assignments?",
  'document': '977bf7786c'},
 {'question': "What's the point of registering if it's not required and I won't get a confirmation?",
  'document': '977bf7786c'}]

In [31]:
import pandas as pd

In [32]:
pd.DataFrame(records)

,question,document
0,I signed up for the LLM course but haven't got...,977bf7786c
1,Do I really need to register for the course to...,977bf7786c
2,I registered but didn't get any email. Should ...,977bf7786c
3,"If I don't register, can I still join the cour...",977bf7786c
4,What's the point of registering if it's not re...,977bf7786c


In [33]:
from evaluation_utils import llm_structured_retry

In [38]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions,
        model="deepseek-v4-flash"
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["doc_id"]
        })

    return results, usage

In [35]:
doc

{'course': 'llm',
 'section': 'General Course-Related Questions',
 'question': 'Course: I have registered for the LLM . When can I expect to receive the confirmation email?',
 'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.",
 'doc_id': '977bf7786c'}

In [36]:
generate_ground_truth(doc)

([{'question': 'Do I need to wait for a digital message telling me I am good to go?',
   'document': '977bf7786c'},
  {'question': "Will checking the records stop me if my entry isn't there yet?",
   'document': '977bf7786c'},
  {'question': 'Is it fine to dive straight into the materials and hand in projects now?',
   'document': '977bf7786c'},
  {'question': 'Are students enrolled by default without needing explicit permission?',
   'document': '977bf7786c'},
  {'question': 'What purpose does filling out that document serve before the term begins?',
   'document': '977bf7786c'}],
 CompletionUsage(completion_tokens=5341, prompt_tokens=282, total_tokens=5623, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=5259, rejected_prediction_tokens=None, text_tokens=5341), prompt_tokens_details=PromptTokensDetails(audio_tokens=None, cached_tokens=None, text_tokens=282)))

In [37]:
documents[:5]

[{'course': 'llm',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'llm',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM . When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.",
  'doc_id': '977bf7786c'},
 {'course': 'llm',
  'section': 'General Course-Related Questions',
  'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?',
  'answer': 'The zoom link is only published to instructors/

In [ ]:
data_gen_instructions

In [39]:
len(documents[:5])

5

In [40]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [41]:
ground_truth

[{'question': 'I found this course late in the semester. Is there still a way to get a certificate if I enroll now?',
  'document': '74eb249bbf'},
 {'question': "What's the deal with the project submission deadline? If I join the course now, can I still submit my project on time?",
  'document': '74eb249bbf'},
 {'question': 'Do I need to complete a project to get the certificate, and if so, is there a cutoff date for submissions?',
  'document': '74eb249bbf'},
 {'question': 'If I join the course after it started, can I still earn a certificate, or is it too late for that?',
  'document': '74eb249bbf'},
 {'question': "I'm interested in taking this course but I missed the beginning. Is it possible to still get the certificate if I submit my project before the submission window closes?",
  'document': '74eb249bbf'},
 {'question': 'Do I need to wait for a confirmation email before starting the LLM course?',
  'document': '977bf7786c'},
 {'question': "I signed up but didn't get any email – 

In [42]:
len(ground_truth)
# pd.DataFrame(ground_truth)

25

In [43]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [ ]:
len(documents)

In [44]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/111 [00:00<?, ?it/s]

In [45]:
len(results)

111

In [ ]:
results

In [46]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

556

In [48]:
ground_truth[10]

{'question': 'Where can I find the YouTube link for the office hours or live workshop sessions?',
 'document': '489dd1c9d9'}

In [ ]:
usage

In [49]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.022784100000000005

In [ ]:
from evaluation_utils import calc_total_price
calc_total_price(usages)

In [50]:
df_ground_truth = pd.DataFrame(ground_truth)

In [51]:
df_ground_truth

,question,document
0,I just stumbled upon this course. Am I still a...,74eb249bbf
1,"If I enroll now, will I be eligible for a cert...",74eb249bbf
2,What's the cutoff date for project submissions...,74eb249bbf
3,I missed the beginning of the course. Can I st...,74eb249bbf
4,Is it possible to join the course late and sti...,74eb249bbf
...,...,...
551,How can I force pip to install a newer version...,4b30b918bc
552,I'm hitting a 401 error with lancedb - does th...,4b30b918bc
553,Why does pip only offer requests 2.28 when I n...,4b30b918bc
554,Can I install requests directly from a GitHub ...,4b30b918bc


In [52]:
df_ground_truth.to_csv("../data/ground_truth-new-G14.csv", index=False)

In [ ]:
len(df_ground_truth)